In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
DEVICE = torch.device("cuda:1")

In [7]:
class RNNPositionalEncoding(nn.Module):
    """
    Positional encoding using a GRU (Gated Recurrent Unit) to generate
    learned, dynamic position encodings, as described in papers relating
    transformers to hippocampal models.

    This module uses an RNN to process a sequence of zero vectors, and its
    hidden states are used as the positional encodings.
    """

    def __init__(
        self,
        max_length: int,
        embed_dim: int,
        hidden_dim: int = None,
        num_layers: int = 1,
    ):
        """
        Initializes the RNNPositionalEncoding module.

        Args:
            max_length (int): The maximum sequence length that this module
                              will be used for. Not used.
            embed_dim (int): The dimensionality of the input embeddings. The
                             positional encodings will also have this dimension.
            hidden_dim (int, optional): The dimensionality of the RNN's hidden
                                        state. If None, it defaults to embed_dim.
            num_layers (int, optional): The number of layers in the RNN.
                                        Defaults to 1.
        """
        super().__init__()
        self.embed_dim = embed_dim
        self.hidden_dim = hidden_dim if hidden_dim is not None else embed_dim
        self.num_layers = num_layers

        # The core RNN (GRU) that learns to generate positional patterns.
        self.rnn = nn.GRU(
            input_size=embed_dim,
            hidden_size=self.hidden_dim,
            num_layers=num_layers,
            batch_first=True,  # Crucial for [B, L, D] input shape
        )
        self.init_input = nn.Parameter(torch.zeros(1, 1, self.embed_dim))
        # If the RNN's hidden dimension is different from the embedding dimension,
        # use linear layer to project it back to the correct size.
        if self.hidden_dim != embed_dim:
            self.proj = nn.Linear(self.hidden_dim, embed_dim)
        else:
            # If dimensions match, no projection is needed.
            self.proj = nn.Identity()

    def forward(self, feat: torch.Tensor):
        """
        Adds positional encoding to a complete input sequence.

        Args:
            feat: Input tensor of shape [B, L, D], where B is the batch size,
                  L is the sequence length, and D is the embedding dimension.

        Returns:
            A tensor of shape [B, L, D] with positional encodings added.
        """
        B, L = feat.shape[0], feat.shape[1]
        # Generate positional encodings for the entire sequence length.
        # initial hidden state, which is a learnable parameter
        h_0 = self.init_input.expand(self.num_layers, B, self.hidden_dim)
        # output shape: [B, L, hidden_dim]
        pos_enc, h_n = self.rnn(feat, h_0)
        # Project the encodings to the correct embedding dimension if necessary.
        pos_enc = self.proj(pos_enc)

        return pos_enc

    def forward_with_position(self, feat, position, last_hidden):
        """
        Adds positional encoding at a specific position. This is useful for
        autoregressive decoding where inputs are processed one at a time.

        Args:
            feat: Input tensor of shape [B, 1, D] for the single item.
            position: The position index (integer) to generate the encoding for.

        Returns:
            A tensor of shape [B, 1, D] with the specific positional encoding added.
        """
        B = feat.shape[0]
        # Generate encodings for all positions up to and including `position`.
        if last_hidden is None:
            last_hidden = self.init_input.expand(self.num_layers, B, self.hidden_dim)
        all_pos_enc, h_n = self.rnn(feat, last_hidden)
        print("DEBUG, all_pos_enc shape:", all_pos_enc.shape)
        # Select the encoding for the specific position we need.
        # The shape becomes [B, 1, D] to match the input `feat`.
        pos_enc_at_position = self.proj(all_pos_enc[:, position : position + 1, :])

        return pos_enc_at_position, h_n


In [8]:
emb_dim = 16
hid_dim = 16
batch_size = 3
seq_length = 8
num_layers = 2
# --- Model Initialization ---
rnn_pos_encoder = RNNPositionalEncoding(
    max_length=None,
    embed_dim=emb_dim,
    hidden_dim=hid_dim,
    num_layers=num_layers,
)


In [20]:
print("Model initialized:")
print(rnn_pos_encoder)

print("\nInitial Hidden State:", rnn_pos_encoder.init_input)


Model initialized:
RNNPositionalEncoding(
  (rnn): GRU(16, 16, num_layers=2, batch_first=True)
  (proj): Identity()
)

Initial Hidden State: Parameter containing:
tensor([[[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]]],
       requires_grad=True)


In [22]:
# --- Test `forward()` method ---
print("\n--- Testing forward() method ---")
# Create a dummy feature tensor
input_features = torch.randn(batch_size, seq_length, emb_dim)
print(f"Input feature shape: {input_features.shape}")
# Get the output with positional encodings
output_features = rnn_pos_encoder(input_features)
print(f"Output feature shape: {output_features.shape}")


--- Testing forward() method ---
Input feature shape: torch.Size([3, 8, 16])
Output feature shape: torch.Size([3, 8, 16])


In [23]:
print("\n--- Testing Breakdown: forward method ---")
rnn = nn.GRU(
    input_size=emb_dim,
    hidden_size=emb_dim,
    num_layers=2,
    batch_first=True,  # Crucial for [B, L, D] input shape
)
init_input = nn.Parameter(torch.zeros(num_layers, 1, emb_dim))

feat = torch.randn(batch_size, seq_length, emb_dim)
print(f"Input feature shape: {feat.shape}")
B, L = feat.shape[0], feat.shape[1]
# Generate positional encodings for the entire sequence length.
# initial hidden state, which is a learnable parameter
h_0 = init_input.expand(num_layers, B, emb_dim)
print(f"Initial hidden state shape: {h_0.shape}")
# output shape: [B, L, hidden_dim]
pos_enc, h_n = rnn(feat, h_0)
print("Positional encodings shape:", pos_enc.shape)
print("Last hidden state shape:", h_n.shape)


--- Testing Breakdown: forward method ---
Input feature shape: torch.Size([3, 8, 16])
Initial hidden state shape: torch.Size([2, 3, 16])
Positional encodings shape: torch.Size([3, 8, 16])
Last hidden state shape: torch.Size([2, 3, 16])


In [ ]:
# --- Test `forward_with_position()` method ---
print("\n--- Testing forward_with_position() method ---")
# Create a dummy feature tensor for a single item
single_feature = torch.randn(batch_size, 1, emb_dim)
target_position = 15 # e.g., we are at the 16th token
print(f"Input single feature shape: {single_feature.shape}")
print(f"Target position: {target_position}")

# Get the output with positional encoding for that specific position
output_single_feature, h_n = rnn_pos_encoder.forward_with_position(single_feature, target_position, None)
print(f"Output single feature shape: {output_single_feature.shape}")




--- Testing forward_with_position() method ---
Input single feature shape: torch.Size([3, 1, 16])
Target position: 15
Output single feature shape: torch.Size([3, 0, 16])


AssertionError: 

In [10]:
raw_input = torch.randn(3, 8, 16)
last_pos = 0
h_0 = None
e_t, h_next = rnn_pos_encoder.forward_with_position(
            raw_input[:,0:last_pos+1,:], last_pos, h_0
        )
print(f"last_pos: {last_pos}, raw_input shape: {raw_input.shape}, e_t shape: {e_t.shape}, h_next shape: {h_next.shape}")

last_pos = 1
e_t, h_next = rnn_pos_encoder.forward_with_position(
            raw_input[:,0:last_pos+1,:], last_pos, h_next)
print(f"last_pos: {last_pos}, raw_input shape: {raw_input.shape}, e_t shape: {e_t.shape}, h_next shape: {h_next.shape}")

last_pos = 2
e_t, h_next = rnn_pos_encoder.forward_with_position(
            raw_input[:,0:last_pos+1,:], last_pos, h_next)
print(f"last_pos: {last_pos}, raw_input shape: {raw_input.shape}, e_t shape: {e_t.shape}, h_next shape: {h_next.shape}")


DEBUG, all_pos_enc shape: torch.Size([3, 1, 16])
last_pos: 0, raw_input shape: torch.Size([3, 8, 16]), e_t shape: torch.Size([3, 1, 16]), h_next shape: torch.Size([2, 3, 16])
DEBUG, all_pos_enc shape: torch.Size([3, 2, 16])
last_pos: 1, raw_input shape: torch.Size([3, 8, 16]), e_t shape: torch.Size([3, 1, 16]), h_next shape: torch.Size([2, 3, 16])
DEBUG, all_pos_enc shape: torch.Size([3, 3, 16])
last_pos: 2, raw_input shape: torch.Size([3, 8, 16]), e_t shape: torch.Size([3, 1, 16]), h_next shape: torch.Size([2, 3, 16])
